In [2]:
# ===============================
# 1. IMPORT LIBRARIES
# ===============================
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# ===============================
# 2. DATASET
# ===============================
data = """artificial intelligence systems learn patterns from data.
sequence models process information step by step.
recurrent neural networks are useful for sequence prediction.
lstm networks handle long term dependencies.

deep learning models improve sequence learning.
generative models create new samples from learned patterns.
language models predict the next word in a sentence.
sequence generation is used in chatbots and assistants.

machine learning helps computers learn automatically.
training data improves model accuracy.
neural networks simulate human brain structures.
optimization algorithms improve learning efficiency.

technology is transforming modern education.
online learning platforms use artificial intelligence.
students benefit from intelligent tutoring systems.
automation improves productivity and decision making."""

# ===============================
# 3. TOKENIZATION
# ===============================
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])

total_words = len(tokenizer.word_index) + 1
print("Total Words:", total_words)

# ===============================
# 4. CREATE SEQUENCES
# ===============================
input_sequences = []

for line in data.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

# ===============================
# 5. PADDING
# ===============================
max_seq_len = max(len(seq) for seq in input_sequences)

input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# One-hot encoding
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# ===============================
# 6. BUILD LSTM MODEL
# ===============================
model = Sequential([
    Embedding(total_words, 64, input_length=max_seq_len-1),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# ===============================
# 7. TRAIN MODEL
# ===============================
model.fit(X, y, epochs=200, verbose=1)

# ===============================
# 8. TEXT GENERATION FUNCTION
# ===============================
def generate_text(seed_text, next_words=5):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')

        predicted = np.argmax(model.predict(token_list), axis=-1)[0]

        for word, index in tokenizer.word_index.items():
            if index == predicted:
                seed_text += " " + word
                break

    return seed_text

# ===============================
# 9. OUTPUT
# ===============================
print(generate_text("artificial intelligence"))
print(generate_text("machine learning"))

Total Words: 78
Epoch 1/200


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 34ms/step - accuracy: 0.0227 - loss: 4.3572
Epoch 2/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.0455 - loss: 4.3461 
Epoch 3/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.0568 - loss: 4.3353
Epoch 4/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.0568 - loss: 4.3218
Epoch 5/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.0568 - loss: 4.3043
Epoch 6/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.0568 - loss: 4.2692
Epoch 7/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.0568 - loss: 4.2211    
Epoch 8/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.0568 - loss: 4.1592 
Epoch 9/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.0568 - loss: 4.1073
Epoch 10/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.0568 - loss: 4.0797
Epoch 11/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.0568 - loss: 4.0460
Epoch 12/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.0568 - loss: 4.

In [3]:
# ===============================
# 1. TOKENIZATION AGAIN
# ===============================
from tensorflow.keras.layers import MultiHeadAttention, LayerNormalization, Dropout

tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])

index_to_word = {index: word for word, index in tokenizer.word_index.items()}

sequences = []
for line in data.split('\n'):
    seq = tokenizer.texts_to_sequences([line])[0]
    sequences.append(seq)

max_len = max(len(seq) for seq in sequences)

sequences = pad_sequences(sequences, maxlen=max_len, padding='post')

X_t = sequences[:, :-1]
y_t = sequences[:, -1]

# ===============================
# 2. POSITIONAL ENCODING
# ===============================
def positional_encoding(position, d_model):
    angles = np.arange(position)[:, np.newaxis] / np.power(10000, (2 * (np.arange(d_model)[np.newaxis, :]//2)) / np.float32(d_model))

    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])

    return tf.cast(angles, dtype=tf.float32)

embed_dim = 32
pos_encoding = positional_encoding(max_len, embed_dim)

# ===============================
# 3. TRANSFORMER BLOCK
# ===============================
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()

        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)

        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim)
        ])

        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)

    def call(self, inputs):
        attn_output = self.att(inputs, inputs)
        out1 = self.layernorm1(inputs + attn_output)

        ffn_output = self.ffn(out1)
        return self.layernorm2(out1 + ffn_output)

# ===============================
# 4. BUILD MODEL
# ===============================
inputs = tf.keras.Input(shape=(max_len - 1,))

x = Embedding(total_words, embed_dim)(inputs)

x = x + pos_encoding[:max_len - 1]

transformer_block = TransformerBlock(embed_dim, 2, 128)
x = transformer_block(x)

x = tf.keras.layers.GlobalAveragePooling1D()(x)

outputs = Dense(total_words, activation="softmax")(x)

transformer_model = tf.keras.Model(inputs=inputs, outputs=outputs)

transformer_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ===============================
# 5. TRAIN
# ===============================
transformer_model.fit(X_t, y_t, epochs=200, verbose=1)

# ===============================
# 6. GENERATE TEXT
# ===============================
def generate_transformer(seed_text, next_words=5):
    for _ in range(next_words):
        seq = tokenizer.texts_to_sequences([seed_text])[0]
        seq = pad_sequences([seq], maxlen=max_len - 1, padding='post')

        pred = np.argmax(transformer_model.predict(seq), axis=-1)[0]

        next_word = index_to_word.get(pred, None)
        if next_word is None:
            break

        seed_text += " " + next_word

    return seed_text

# ===============================
# 7. OUTPUT
# ===============================
print(generate_transformer("deep learning"))
print(generate_transformer("sequence models"))

Epoch 1/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.0000e+00 - loss: 4.9478
Epoch 2/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.0000e+00 - loss: 4.3291
Epoch 3/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.0000e+00 - loss: 3.7306
Epoch 4/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.0000e+00 - loss: 3.1842
Epoch 5/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.9474 - loss: 2.7050
Epoch 6/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.9474 - loss: 2.2986
Epoch 7/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.9474 - loss: 1.9624
Epoch 8/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.9474 - loss: 1.6911
Epoch 9/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.9474 - loss: 1.4751
Epoch 10/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.9474 - loss: 1.3022
Epoch 11/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.9474 - loss: 1.1638
Epoch 12/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accurac